# Phase 2 — Multi-seed comparison & reward ablation

Goal of this notebook:

1. Train PPO and DQN at 5 seeds each (using the default `diff-waiting-time` reward)
2. Train PPO at 3 seeds for each of the 3 alternative reward functions
3. Evaluate every trained model on 5 held-out SUMO seeds
4. Aggregate results: PPO vs DQN table, reward ablation plot

**Note on runtime:** each training takes ~2–3h on Windows (TraCI), ~15min on Linux (libsumo).
Training cells are restartable — they skip models that already exist on disk.

---

## Setup

In [1]:
import ppo
import dqn
from evaluate import evaluate

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from stable_baselines3 import PPO, DQN

# Configuration
SEEDS = [42, 43, 44, 45, 46]            # training seeds for PPO vs DQN comparison
ABLATION_SEEDS = [42, 43, 44]            # training seeds for the reward ablation
REWARDS = ["pressure", "queue", "average-speed"]   # alternative rewards (default is diff-waiting-time)
EVAL_SEEDS = [100, 200, 300, 400, 500]   # held-out SUMO seeds for evaluation

## 1. Training

### 1a. Multi-seed: PPO and DQN with the default reward

In [ ]:
for seed in SEEDS:
    ppo.train(seed=seed)
    dqn.train(seed=seed)

### 1b. Reward function ablation (PPO only)

In [ ]:
for reward in REWARDS:
    for seed in ABLATION_SEEDS:
        ppo.train(seed=seed, reward_fn=reward)

## 2. Evaluation

Load every trained model and evaluate it on 5 SUMO seeds. Combine everything into one tidy DataFrame.

In [ ]:
def load_and_eval(algo, seed, reward_fn="diff-waiting-time"):
    """Load one model and run evaluate() on it. Returns DataFrame with metadata columns added."""
    tag = f"seed{seed}"
    if reward_fn != "diff-waiting-time":
        tag += f"_{reward_fn}"
    model_path = Path(f"checkpoints/cologne1/{algo}/{tag}/final.zip")
    if not model_path.exists():
        print(f"  missing: {model_path}")
        return None
    cls = PPO if algo == "ppo" else DQN
    print(f"Evaluating {algo} {tag}")
    model = cls.load(str(model_path))
    df = evaluate(model, eval_seeds=EVAL_SEEDS, reward_fn=reward_fn)
    df["algo"] = algo
    df["train_seed"] = seed
    df["reward_fn"] = reward_fn
    return df


frames = []

# Multi-seed runs
for seed in SEEDS:
    for algo in ["ppo", "dqn"]:
        df = load_and_eval(algo, seed)
        if df is not None:
            frames.append(df)

# Reward ablation runs (PPO only)
for reward in REWARDS:
    for seed in ABLATION_SEEDS:
        df = load_and_eval("ppo", seed, reward_fn=reward)
        if df is not None:
            frames.append(df)

results = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(results)} eval rows across {results['train_seed'].nunique()} training seeds")
results.head()

## 3. Results

### 3a. Summary table — mean ± std across training seeds

In [ ]:
summary = results.groupby(["algo", "reward_fn"]).agg(
    n_seeds=("train_seed", "nunique"),
    wait_mean=("mean_wait", "mean"),
    wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"),
    speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"),
    stopped_std=("mean_stopped", "std"),
    p95_mean=("p95_wait", "mean"),
    p95_std=("p95_wait", "std"),
).round(3)

summary

### 3b. PPO vs DQN (on the default reward)

In [ ]:
comp = results[results["reward_fn"] == "diff-waiting-time"]
agg = comp.groupby("algo").agg(
    wait_mean=("mean_wait", "mean"),
    wait_std=("mean_wait", "std"),
    speed_mean=("mean_speed", "mean"),
    speed_std=("mean_speed", "std"),
    stopped_mean=("mean_stopped", "mean"),
    stopped_std=("mean_stopped", "std"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {"ppo": "#0ea5e9", "dqn": "#e8772e"}

for ax, (col_mean, col_std, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean wait (s)"),
    ("speed_mean", "speed_std", "Mean speed (m/s)"),
    ("stopped_mean", "stopped_std", "Avg stopped vehicles"),
]):
    bar_colors = [colors[a] for a in agg["algo"]]
    ax.bar(agg["algo"].str.upper(), agg[col_mean], yerr=agg[col_std],
           color=bar_colors, capsize=8, edgecolor="black", linewidth=0.5)
    for i, (m, s) in enumerate(zip(agg[col_mean], agg[col_std])):
        ax.text(i, m + s + max(agg[col_mean]) * 0.03, f"{m:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(f"PPO vs DQN on Cologne1 ({len(SEEDS)} training seeds, mean ± std)", fontweight="bold")
fig.tight_layout()
plt.show()

### 3c. Reward function ablation (PPO)

In [ ]:
ppo_results = results[results["algo"] == "ppo"]
abl = ppo_results.groupby("reward_fn").agg(
    wait_mean=("mean_wait", "mean"),
    wait_std=("mean_wait", "std"),
    p95_mean=("p95_wait", "mean"),
    p95_std=("p95_wait", "std"),
    n_seeds=("train_seed", "nunique"),
).reset_index().sort_values("wait_mean")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (col_mean, col_std, label) in zip(axes, [
    ("wait_mean", "wait_std", "Mean waiting time (s) — lower is better"),
    ("p95_mean",  "p95_std",  "p95 waiting time (s) — tail / fairness"),
]):
    ax.bar(abl["reward_fn"], abl[col_mean], yerr=abl[col_std],
           color="#0ea5e9", capsize=8, edgecolor="black", linewidth=0.5)
    for i, (m, s) in enumerate(zip(abl[col_mean], abl[col_std])):
        ax.text(i, m + s + max(abl[col_mean]) * 0.03, f"{m:.2f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(label)
    ax.set_xticklabels(abl["reward_fn"], rotation=15, ha="right")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("PPO reward-function ablation on Cologne1", fontweight="bold")
fig.tight_layout()
plt.show()

abl

### 3d. Save the summary table for the next report

In [ ]:
out_path = Path("outputs/cologne1/phase2_summary.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_path)
print(f"Saved -> {out_path}")